# Live Liver Donor Expert System
Rule-based screening followed by a **multi-model** ML vote (KNN, Logistic Regression, Naive Bayes, Decision Tree).

## Model 1: KNN + data (Lab 1)

In [1]:
import numpy as np

# Historical Liver Donor Dataset
# Features: [Age, BMI, Smoking Habit (0=No, 1=Yes), General Health Score (1-5)]
X_train = np.array([
    [28, 22.5, 0, 5],
    [55, 34.0, 1, 2],
    [35, 24.0, 0, 4],
    [45, 31.5, 1, 2],
    [24, 21.0, 0, 5],
    [58, 29.0, 0, 3],
    [40, 36.0, 1, 1],
    [30, 23.5, 0, 4],
])
y_train = np.array([1, 0, 1, 0, 1, 1, 0, 1])

class Scaler:
    def fit(self, X):
        self.mean = np.mean(X, axis=0)
        self.std = np.std(X, axis=0)
        self.std[self.std == 0] = 1e-8

    def transform(self, X):
        return (X - self.mean) / self.std

    def fit_transform(self, X):
        self.fit(X)
        return self.transform(X)

class KNN:

  def __init__(self, k=3):
    self.k = k
    self.scaler = Scaler()

  def fit(self, X, y):
    self.X_train = self.scaler.fit_transform(X)
    self.y_train = y


  def _euclidean_distance(self, x1, x2):
    return np.sqrt(np.sum((x1 - x2) ** 2))

  def predict(self, x_new):
    distances = []
    x_new = self.scaler.transform(x_new)
    for x_train in self.X_train:
      dist = self._euclidean_distance(x_new, x_train)
      distances.append(dist)

    k_indices = np.argsort(distances)[: self.k]
    k_nearest_labels = self.y_train[k_indices]

    unique_labels, counts = np.unique(k_nearest_labels, return_counts=True)
    return unique_labels[np.argmax(counts)]


knn = KNN(k=3)
knn.fit(X_train, y_train)
print('System Initialized: Historical data loaded & KNN model ready.')

System Initialized: Historical data loaded & KNN model ready.


## Model 2: Logistic Regression (Lab 4)

In [2]:
class LogisticRegression:
    """Logistic regression trained with gradient descent + L2 regularization (from Lab 4)."""

    def __init__(self, lr=0.1, alpha=1.0, n_iter=5000):
        self.lr = lr
        self.alpha = alpha
        self.n_iter = n_iter
        self.scaler = Scaler()

    @staticmethod
    def _sigmoid(z):
        return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

    def fit(self, X, y):
        X = self.scaler.fit_transform(np.asarray(X, dtype=float))
        y = np.asarray(y, dtype=float)
        N = len(X)
        self.w = np.zeros(X.shape[1])
        self.b = 0.0
        for _ in range(self.n_iter):
            p = self._sigmoid(X @ self.w + self.b)
            e = p - y
            db = e.sum() / N
            dw = (X.T @ e) / N + (self.alpha / N) * self.w
            self.b -= self.lr * db
            self.w -= self.lr * dw

    def predict_proba(self, x):
        x = self.scaler.transform(np.asarray(x, dtype=float))
        return float(self._sigmoid(x @ self.w + self.b))

    def predict(self, x, thr=0.5):
        return int(self.predict_proba(x) >= thr)


logreg = LogisticRegression()
logreg.fit(X_train, y_train)
print('Logistic Regression trained. Feature weights (standardized):',
      np.round(logreg.w, 2))

Logistic Regression trained. Feature weights (standardized): [-0.11 -0.61 -0.94  0.61]


## Models 3 & 4: Naive Bayes (Lab 5) and Decision Tree (Lab 2)
These two work on categories, so the numeric features are first grouped into bins.

In [3]:
FEATURE_NAMES = ['Age', 'BMI', 'Smoking', 'Health']

# Human-readable names for each category (used when explaining the tree)
CATEGORY_LABELS = {
    'Age':     {1: '<=30', 2: '31-45', 3: '>45'},
    'BMI':     {1: '<25 (normal)', 2: '25-30 (overweight)', 3: '>30 (obese)'},
    'Smoking': {0: 'non-smoker', 1: 'smoker'},
    'Health':  {1: 'poor (1-2)', 2: 'average (3)', 3: 'good (4-5)'},
}

def categorize_features(sample):
    """Turns [age, bmi, smoking, health] into categories (as in Lab 1)."""
    age, bmi, smoking, health = sample

    cat_age = 1 if age <= 30 else 2 if age <= 45 else 3
    cat_bmi = 1 if bmi < 25 else 2 if bmi <= 30 else 3
    cat_health = 1 if health <= 2 else 2 if health == 3 else 3

    return [cat_age, cat_bmi, int(smoking), cat_health]

X_train_cat = [categorize_features(x) for x in X_train]

In [4]:
from collections import Counter
import math

class CategoricalNaiveBayes:
    """Naive Bayes for categorical features with Laplace smoothing (from Lab 5)."""

    def __init__(self, alpha=1.0):
        self.alpha = alpha

    def fit(self, X, y):
        y = [int(v) for v in y]
        n_features = len(X[0])
        self.classes = sorted(set(y))
        self.class_counts = Counter(y)
        self.priors = {c: self.class_counts[c] / len(y) for c in self.classes}
        self.values = [set(row[j] for row in X) for j in range(n_features)]
        self.counts = {c: [Counter() for _ in range(n_features)] for c in self.classes}
        for row, label in zip(X, y):
            for j, v in enumerate(row):
                self.counts[label][j][v] += 1

    def _likelihood(self, c, j, v):
        numerator = self.counts[c][j][v] + self.alpha
        denominator = self.class_counts[c] + self.alpha * len(self.values[j])
        return numerator / denominator

    def predict_proba(self, row):
        """Returns {class: posterior probability}."""
        log_scores = {}
        for c in self.classes:
            score = math.log(self.priors[c])
            for j, v in enumerate(row):
                score += math.log(self._likelihood(c, j, v))
            log_scores[c] = score
        m = max(log_scores.values())
        exp = {c: math.exp(s - m) for c, s in log_scores.items()}
        total = sum(exp.values())
        return {c: v / total for c, v in exp.items()}

    def predict(self, row):
        proba = self.predict_proba(row)
        return max(proba, key=proba.get)


nb = CategoricalNaiveBayes()
nb.fit(X_train_cat, y_train)
print('Naive Bayes trained on categorized features.')

Naive Bayes trained on categorized features.


In [5]:
class DecisionTree:
    """ID3 decision tree using entropy / information gain (from Lab 2)."""

    def __init__(self, max_depth=None):
        self.max_depth = max_depth

    @staticmethod
    def _entropy(y):
        n = len(y)
        return -sum((c / n) * math.log2(c / n) for c in Counter(y).values())

    def _gain(self, X, y, j):
        n = len(y)
        remainder = 0.0
        for v in set(row[j] for row in X):
            y_v = [label for row, label in zip(X, y) if row[j] == v]
            remainder += (len(y_v) / n) * self._entropy(y_v)
        return self._entropy(y) - remainder

    def _build(self, X, y, features, depth):
        majority = Counter(y).most_common(1)[0][0]
        if len(set(y)) == 1 or not features or (self.max_depth is not None and depth >= self.max_depth):
            return {'leaf': majority}

        best = max(features, key=lambda j: self._gain(X, y, j))
        children = {}
        for v in set(row[best] for row in X):
            idx = [i for i, row in enumerate(X) if row[best] == v]
            children[v] = self._build(
                [X[i] for i in idx], [y[i] for i in idx],
                [f for f in features if f != best], depth + 1,
            )
        return {'feature': best, 'majority': majority, 'children': children}

    def fit(self, X, y):
        y = [int(v) for v in y]
        self.root = self._build(X, y, list(range(len(X[0]))), 0)

    def predict(self, row):
        node = self.root
        while 'leaf' not in node:
            v = row[node['feature']]
            if v not in node['children']:   # category never seen in this branch
                return node['majority']
            node = node['children'][v]
        return node['leaf']

    def print_tree(self, node=None, indent=''):
        node = node or self.root
        if 'leaf' in node:
            print(f"{indent}-> {'ELIGIBLE' if node['leaf'] == 1 else 'HIGH RISK'}")
            return
        name = FEATURE_NAMES[node['feature']]
        for v, child in sorted(node['children'].items()):
            print(f"{indent}{name} = {CATEGORY_LABELS[name][v]}")
            self.print_tree(child, indent + '    ')


tree = DecisionTree()
tree.fit(X_train_cat, y_train)
print('Decision Tree trained. Learned rules:')
tree.print_tree()

Decision Tree trained. Learned rules:
BMI = <25 (normal)
    -> ELIGIBLE
BMI = 25-30 (overweight)
    -> ELIGIBLE
BMI = >30 (obese)
    -> HIGH RISK


## Rule-based screening

In [6]:
def evaluate_liver_donor_rules(
    age, bmi, smoking, has_infection, has_chronic_disease
):
  """Applies mandatory medical exclusion criteria for live liver donation."""

  if age < 18 or age > 60:
    return (
        False,
        f'Rejected by Rules: Age {age} is outside the safe range (18-60 years).'
        ' Live donors generally need to be adults under 60.',
    )

  if bmi < 18.5 or bmi > 32.0:
    return (
        False,
        f'Rejected by Rules: BMI {bmi:.1f} is outside the acceptable range'
        ' (18.5 - 32.0).',
    )

  if has_infection:
    return (
        False,
        'Rejected by Rules: Active infection detected. Cannot proceed while'
        ' fighting an infection.',
    )

  if has_chronic_disease:
    return (
        False,
        'Rejected by Rules: History of chronic disease disqualifies live'
        ' organ donation.',
    )

  return True, 'Passed: All hard medical screening filters cleared.'

## Multi-model prediction

In [7]:
MODEL_NAMES = ['KNN (k=3)', 'Logistic Regression', 'Naive Bayes', 'Decision Tree']

def multi_model_predict(age, bmi, smoking, health_score):
    """Asks every ML model for its opinion and combines them by majority vote."""
    x = np.array([age, bmi, smoking, health_score], dtype=float)
    x_cat = categorize_features([age, bmi, smoking, health_score])

    votes = {
        'KNN (k=3)': int(knn.predict(x)),
        'Logistic Regression': logreg.predict(x),
        'Naive Bayes': int(nb.predict(x_cat)),
        'Decision Tree': int(tree.predict(x_cat)),
    }
    # Probability of "eligible" where the model provides one
    p_eligible = {
        'Logistic Regression': logreg.predict_proba(x),
        'Naive Bayes': nb.predict_proba(x_cat)[1],
    }

    n_yes = sum(votes.values())
    n_total = len(votes)
    if n_yes > n_total / 2:
        verdict = 'ELIGIBLE'
    elif n_yes < n_total / 2:
        verdict = 'HIGH_RISK'
    else:
        verdict = 'INCONCLUSIVE'   # tie: safest option is a human decision

    return {'votes': votes, 'p_eligible': p_eligible,
            'n_yes': n_yes, 'n_total': n_total, 'verdict': verdict}


def print_ensemble_report(result):
    print('\n--- ML MODEL BLOCK (multi-model vote) ---')
    for name in MODEL_NAMES:
        vote = 'eligible' if result['votes'][name] == 1 else 'high risk'
        extra = ''
        if name in result['p_eligible']:
            extra = f"  (P(eligible) = {result['p_eligible'][name]:.0%})"
        print(f'  {name:<20} -> {vote}{extra}')
    print(f"\nVotes for eligible: {result['n_yes']} / {result['n_total']}")

    unanimous = result['n_yes'] in (0, result['n_total'])
    if result['verdict'] == 'ELIGIBLE':
        print(
            'Result: 🟢 ELIGIBLE DONOR PROFILE. Based on historical data similarity,'
            ' your profile matches successful live liver donors. (Note: Clinical'
            ' lab tests are still required).'
        )
    elif result['verdict'] == 'HIGH_RISK':
        print(
            'Result: 🔴 HIGH RISK PROFILE. While you passed initial safety rules,'
            ' historical patterns suggest potential elevated risks for this type'
            ' of donation profile.'
        )
    else:
        print(
            'Result: 🟡 INCONCLUSIVE. The models are split 50/50 on this profile,'
            ' so it should be reviewed by a medical professional.'
        )
    if not unanimous and result['verdict'] != 'INCONCLUSIVE':
        print('(The models did not all agree, so treat this result with extra caution.)')


def assess_donor(age, weight_kg, height_m, smoking, has_infection,
                 has_chronic_disease, health_score):
    """Full pipeline: rule-based screening first, then the ML models."""
    bmi = weight_kg / (height_m ** 2)
    print(f'\n--- Calculated BMI: {bmi:.1f} ---')

    passed_rules, message = evaluate_liver_donor_rules(
        age, bmi, smoking, has_infection, has_chronic_disease
    )
    print(f'If-Else Block Status: {message}')

    if not passed_rules:
        print(
            '\nProcess Terminated: Match discarded at the rule-based medical'
            ' screening stage.'
        )
        return None

    result = multi_model_predict(age, bmi, smoking, health_score)
    print_ensemble_report(result)
    return result

## Questionnaire

In [8]:
def ask_yes_no(prompt):
    """Keeps asking until the answer is exactly yes or no."""
    while True:
        answer = input(prompt).strip().lower()
        if answer in ('yes', 'y'):
            return True
        if answer in ('no', 'n'):
            return False
        print("  Please answer 'yes' or 'no'.")


print('=== LIVE LIVER DONOR ELIGIBILITY QUESTIONNAIRE ===\n')

try:
  age = int(input('Enter your age (years): '))
  weight_kg = float(input('Enter your weight (kg): '))
  height_m = float(input('Enter your height (meters, e.g., 1.75): '))
  if weight_kg <= 0 or height_m <= 0:
    raise ValueError

  smoking = 1 if ask_yes_no('Do you smoke regularly? (yes/no): ') else 0
  has_infection = ask_yes_no('Do you have any active infections currently? (yes/no): ')
  has_chronic_disease = ask_yes_no('Do you have chronic diseases (diabetes, hepatitis)? (yes/no): ')

  health_score = int(
      input('Rate your overall self-reported health from 1 (poor) to 5 (excellent): ')
  )
  if not 1 <= health_score <= 5:
    raise ValueError

  assess_donor(age, weight_kg, height_m, smoking, has_infection,
               has_chronic_disease, health_score)

except ValueError:
  print(
      '\nInvalid input format. Please restart the cell and enter numbers where'
      ' requested (health score must be between 1 and 5).'
  )

=== LIVE LIVER DONOR ELIGIBILITY QUESTIONNAIRE ===

Enter your age (years): 22
Enter your weight (kg): 65
Enter your height (meters, e.g., 1.75): 1.70
Do you smoke regularly? (yes/no): no
Do you have any active infections currently? (yes/no): no
Do you have chronic diseases (diabetes, hepatitis)? (yes/no): no
Rate your overall self-reported health from 1 (poor) to 5 (excellent): 4

--- Calculated BMI: 22.5 ---
If-Else Block Status: Passed: All hard medical screening filters cleared.

--- ML MODEL BLOCK (multi-model vote) ---
  KNN (k=3)            -> eligible
  Logistic Regression  -> eligible  (P(eligible) = 94%)
  Naive Bayes          -> eligible  (P(eligible) = 100%)
  Decision Tree        -> eligible

Votes for eligible: 4 / 4
Result: 🟢 ELIGIBLE DONOR PROFILE. Based on historical data similarity, your profile matches successful live liver donors. (Note: Clinical lab tests are still required).
